# INC-001 Cross-source profiling and design options

Read-only; proposes, decides nothing. Joins coverage across all sources, evaluates day-assignment options under the
fixed local timezone **UTC-05:00**, measures completeness per participant-day, and computes the figures behind the G1
design options (week boundary, cohort definition, daily freshness, walking-skeleton source).
Minute files are streamed one participant at a time; only per-participant-day counters are cached.
Printed output is counts, aggregates and participant/date identifiers only. Cell ids are `xs-NN`.

In [1]:
import csv, json, re, time, collections
from datetime import date, datetime, timedelta
from pathlib import Path

T0 = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "participants").is_dir())
DATA = ROOT / "data"
EVENTS = DATA / "wearable_events"
WIN_START, WIN_END = date(2026, 1, 8), date(2026, 2, 4)
DAYS = [WIN_START + timedelta(days=i) for i in range(28)]
ORD0 = WIN_START.toordinal()
N_SLOTS = 28 * 1440
UTC_OFFSET_MIN = -300            # local = UTC - 5h
TS = re.compile(r"^(\d{4})-(\d\d)-(\d\d)T(\d\d):(\d\d):(\d\d)(\.\d+)?(Z|[+-]\d\d:\d\d)?$")

def show(label, value, examples=None):
    extra = f"  examples={sorted(examples)[:5]}" if examples else ""
    print(f"{label}: {value}{extra}")

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

participants_rows = read_csv(DATA / "participants" / "participants.csv")
survey_rows = read_csv(DATA / "health_summaries" / "wellness_survey.csv")
devices_rows = read_csv(DATA / "health_summaries" / "device_metadata.csv")
P = sorted(r["participant_id"] for r in participants_rows)
chrono = {r["participant_id"]: r["chronotype"] for r in participants_rows}
folders = sorted(p.name for p in EVENTS.iterdir() if p.is_dir())
survey_days = collections.defaultdict(set)
for r in survey_rows:
    survey_days[r["participant_id"]].add(r["date"])

## 1. Stream minute files and sleep once; cache compact per-participant-day aggregates

* `naive` = records with timezone-less timestamps, assumed to be local UTC-05:00 as the dictionary states.
* `with_z` = additionally accepts `Z` records after converting UTC to local (a proposal being evaluated, not a decision).
* Quality-flag days (for policy sizing only): duplicate minute slot, steps < 0, heart rate outside 40-200, heart-rate run >= 60 identical values, or `_unit` present.

In [2]:
def scan(pid, fname, vkey):
    rows = json.loads((EVENTS / pid / fname).read_bytes())
    seen = bytearray(N_SLOTS)
    seen_z = bytearray(N_SLOTS)
    flag_days = collections.defaultdict(set)      # reason -> set of day indexes
    z_label_vs_local_day_differs = z_records = 0
    run_val = run_start = None; run_len = 0
    def close_run():
        if run_len >= 60 and run_start is not None:
            for s in range(run_start, min(run_start + run_len, N_SLOTS)):
                flag_days["stuck_run>=60"].add(s // 1440)
    for r in rows:
        m = TS.match(r["timestamp"])
        y, mo, d, hh, mi, ss, frac, tz = m.groups()
        base = (date(int(y), int(mo), int(d)).toordinal() - ORD0) * 1440 + int(hh) * 60 + int(mi)
        v = r.get(vkey)
        if tz == "Z":
            z_records += 1
            loc = base + UTC_OFFSET_MIN
            if loc // 1440 != base // 1440:
                z_label_vs_local_day_differs += 1
            if 0 <= loc < N_SLOTS:
                seen_z[loc] = 1
            continue
        if not 0 <= base < N_SLOTS:
            continue
        if seen[base]:
            flag_days["duplicate_slot"].add(base // 1440)
        seen[base] = 1
        if "_unit" in r:
            flag_days["_unit_field"].add(base // 1440)
        if isinstance(v, int):
            if vkey == "steps" and v < 0:
                flag_days["steps<0"].add(base // 1440)
            if vkey == "heart_rate_bpm":
                if v < 40 or v > 200:
                    flag_days["hr_out_of_range"].add(base // 1440)
                if v == run_val:
                    run_len += 1
                else:
                    close_run(); run_val, run_start, run_len = v, base, 1
    if vkey == "heart_rate_bpm":
        close_run()
    del rows
    naive_counts = [sum(seen[i * 1440:(i + 1) * 1440]) for i in range(28)]
    z_counts = [sum(1 for k in range(i * 1440, (i + 1) * 1440) if seen_z[k] and not seen[k]) for i in range(28)]
    # share of present naive minutes whose local hour < 5 (would roll to previous date if naive were really UTC)
    before5 = sum(1 for i in range(N_SLOTS) if seen[i] and (i % 1440) < 300)
    return {"naive": naive_counts, "z_extra": z_counts, "flags": {k: sorted(v) for k, v in flag_days.items()},
            "z_records": z_records, "z_day_shift": z_label_vs_local_day_differs, "before5": before5,
            "slots": sum(naive_counts), "seen": seen}

def to_secs(ts):
    return int((datetime.fromisoformat(ts) - datetime(2026, 1, 8)).total_seconds())

MIN = {"steps": {}, "heart_rate": {}}
SLEEP = {}
SEEN = {}
for pid in P:
    for kind, vkey in (("steps", "steps"), ("heart_rate", "heart_rate_bpm")):
        res = scan(pid, f"{kind}.json", vkey)
        SEEN_ARR = res.pop("seen")
        MIN[kind][pid] = res
        if kind == "heart_rate":
            SEEN = SEEN_ARR            # only heart-rate presence kept for the current participant
    sessions = json.loads((EVENTS / pid / "sleep.json").read_bytes())
    SLEEP[pid] = []
    for s in sessions:
        on, end = to_secs(s["sleep_onset"]), to_secs(s["sleep_end"])
        last_stage_end = to_secs(s["stages"][-1]["end_time"])
        # whole minutes of heart-rate coverage inside [onset, sleep_end)
        first_min, last_min = -(-on // 60), end // 60          # ceil / floor in minutes
        total = max(0, last_min - first_min)
        covered = sum(SEEN[k] for k in range(max(first_min, 0), min(last_min, N_SLOTS)))
        SLEEP[pid].append({"date": s["date"], "on": on, "end": end, "last_end": last_stage_end, "hr_cov_min": covered, "span_min": total})
    del sessions
print("streamed participants:", len(P), "| elapsed", round(time.perf_counter() - T0, 1), "s")

streamed participants: 50 | elapsed 8.7 s


## 2. Join coverage between sources and study-window bounds

In [3]:
sets = {
    "participants.csv": set(P),
    "device_metadata.csv": {r["participant_id"] for r in devices_rows},
    "wellness_survey.csv": set(survey_days),
    "wearable_events folders": set(folders),
    "sleep.json": set(SLEEP),
    "steps.json": set(MIN["steps"]),
    "heart_rate.json": set(MIN["heart_rate"]),
}
for name, s in sets.items():
    print(f"{name}: participants={len(s)} not in participants.csv={sorted(s - set(P))} missing from this source={sorted(set(P) - s)[:5]}")
survey_in = {p: {d for d in survey_days[p] if WIN_START.isoformat() <= d <= WIN_END.isoformat()} for p in survey_days}
sleep_dates = {p: {x["date"] for x in SLEEP[p]} for p in P}
show("participant-days in survey (enrolled participants only, in window)", sum(len(survey_in.get(p, ())) for p in P))
show("participant-days with a sleep session (wake date in window)", sum(len({d for d in sleep_dates[p] if WIN_START.isoformat() <= d <= WIN_END.isoformat()}) for p in P))
show("participant-days where survey and sleep both present", sum(len(survey_in.get(p, set()) & sleep_dates[p]) for p in P))
show("sleep participant-days without survey row", sum(len(sleep_dates[p] - survey_in.get(p, set())) for p in P))
show("survey participant-days without sleep session (enrolled)", sum(len(survey_in.get(p, set()) - sleep_dates[p]) for p in P))
onset_dates = collections.Counter()
for p in P:
    for x in SLEEP[p]:
        onset_dates[(datetime(2026, 1, 8) + timedelta(seconds=x["on"])).date()] += 1
print("sleep onset date range:", min(onset_dates), "to", max(onset_dates), "| sessions with onset date before study start:", sum(c for d, c in onset_dates.items() if d < WIN_START))
print("sleep wake-date range:", min(x["date"] for p in P for x in SLEEP[p]), "to", max(x["date"] for p in P for x in SLEEP[p]))
print("survey date range (enrolled):", min(d for p in P for d in survey_days[p]), "to", max(d for p in P for d in survey_days[p]))
print("minute-grid records outside the 28-day window (naive): none beyond the grid by construction of the streaming scan (see shr-06 for counts)")
for kind in MIN:
    print(f"{kind}: records on grid per participant min/max = {min(m['slots'] for m in MIN[kind].values())}/{max(m['slots'] for m in MIN[kind].values())}")

participants.csv: participants=50 not in participants.csv=[] missing from this source=[]
device_metadata.csv: participants=50 not in participants.csv=[] missing from this source=[]
wellness_survey.csv: participants=51 not in participants.csv=['P999'] missing from this source=[]
wearable_events folders: participants=50 not in participants.csv=[] missing from this source=[]
sleep.json: participants=50 not in participants.csv=[] missing from this source=[]
steps.json: participants=50 not in participants.csv=[] missing from this source=[]
heart_rate.json: participants=50 not in participants.csv=[] missing from this source=[]
participant-days in survey (enrolled participants only, in window): 1400
participant-days with a sleep session (wake date in window): 1400
participant-days where survey and sleep both present: 1400
sleep participant-days without survey row: 0
survey participant-days without sleep session (enrolled): 0
sleep onset date range: 2026-01-07 to 2026-02-04 | sessions with ons

## 3. Day assignment options under UTC-05:00

Options for minute data: (M1) calendar date of the naive local timestamp; (M2) same plus `Z` records converted from UTC to local first.
Options for sleep: (S1) `date` field (wake date); (S2) calendar date of `sleep_onset`; (S3) calendar date holding most of the session span; (S4) date of `sleep_end`.

In [4]:
def d_of(sec):
    return (datetime(2026, 1, 8) + timedelta(seconds=sec)).date()

agree = collections.Counter(); cross_midnight = 0; onset_collide = 0; total = 0
per_day_option = {"S1": collections.Counter(), "S2": collections.Counter(), "S3": collections.Counter(), "S4": collections.Counter()}
for p in P:
    for x in SLEEP[p]:
        total += 1
        wake = date.fromisoformat(x["date"])
        s1, s2, s4 = wake, d_of(x["on"]), d_of(x["end"])
        # S3: date with the larger share of [onset, end)
        mid_night = datetime.combine(d_of(x["end"]), datetime.min.time())
        mid = int((mid_night - datetime(2026, 1, 8)).total_seconds())
        before, after = max(0, min(x["end"], mid) - x["on"]), max(0, x["end"] - max(x["on"], mid))
        s3 = s2 if before > after else s4
        for k, v in (("S1", s1), ("S2", s2), ("S3", s3), ("S4", s4)):
            per_day_option[k][(p, v)] += 1
        agree["S1==S2"] += s1 == s2; agree["S1==S3"] += s1 == s3; agree["S1==S4"] += s1 == s4
        cross_midnight += s2 != s4
print(f"sleep sessions: {total}")
print("agreement with wake-date S1:", dict(agree))
print("sessions whose onset and end fall on different calendar dates (cross midnight):", cross_midnight)
for k, c in per_day_option.items():
    in_win = sum(1 for (p, d) in c if WIN_START <= d <= WIN_END)
    collide = sum(1 for n in c.values() if n > 1)
    print(f"{k}: distinct participant-days={len(c)} inside window={in_win} outside window={len(c) - in_win} collisions(>1 session on a day)={collide}")
for kind in MIN:
    z = sum(m["z_records"] for m in MIN[kind].values()); sh = sum(m["z_day_shift"] for m in MIN[kind].values())
    print(f"{kind}: Z records={z}; of which local (UTC-05:00) date differs from the label's date: {sh}")
    print(f"{kind}: naive records with local time before 05:00 (would move to previous date if naive were actually UTC): {sum(m['before5'] for m in MIN[kind].values()):,} "
          f"of {sum(m['slots'] for m in MIN[kind].values()):,} ({100*sum(m['before5'] for m in MIN[kind].values())/sum(m['slots'] for m in MIN[kind].values()):.1f}%)")
cov_full = sum(1 for p in P for x in SLEEP[p] if x["span_min"] and x["hr_cov_min"] == x["span_min"])
cov_before = sum(1 for p in P for x in SLEEP[p] if x["on"] < 0)
print(f"sleep sessions with complete heart-rate coverage over [onset, sleep_end): {cov_full} of {total}; sessions starting before the minute grid (onset < 2026-01-08T00:00): {cov_before}")
print("sessions where the final stage extends past sleep_end (final awake stage after wake):", sum(1 for p in P for x in SLEEP[p] if x["last_end"] > x["end"]))

sleep sessions: 1400
agreement with wake-date S1: {'S1==S2': 155, 'S1==S3': 1392, 'S1==S4': 1400}
sessions whose onset and end fall on different calendar dates (cross midnight): 1245
S1: distinct participant-days=1400 inside window=1400 outside window=0 collisions(>1 session on a day)=0
S2: distinct participant-days=1267 inside window=1224 outside window=43 collisions(>1 session on a day)=133
S3: distinct participant-days=1393 inside window=1393 outside window=0 collisions(>1 session on a day)=7
S4: distinct participant-days=1400 inside window=1400 outside window=0 collisions(>1 session on a day)=0
steps: Z records=0; of which local (UTC-05:00) date differs from the label's date: 0
steps: naive records with local time before 05:00 (would move to previous date if naive were actually UTC): 418,800 of 2,009,160 (20.8%)
heart_rate: Z records=2880; of which local (UTC-05:00) date differs from the label's date: 600
heart_rate: naive records with local time before 05:00 (would move to previou

## 4. Completeness per participant-day

Complete minute day = all 1,440 minute slots present (M1 naive only, or M2 including converted `Z` records). A "complete participant-day" also needs a sleep session (S1) and a survey row.

In [5]:
def complete_flags(p, with_z):
    out = []
    for i, day in enumerate(DAYS):
        s_ok = MIN["steps"][p]["naive"][i] + (MIN["steps"][p]["z_extra"][i] if with_z else 0) >= 1440
        h_ok = MIN["heart_rate"][p]["naive"][i] + (MIN["heart_rate"][p]["z_extra"][i] if with_z else 0) >= 1440
        sl = day.isoformat() in sleep_dates[p]
        sv = day.isoformat() in survey_in.get(p, set())
        out.append((s_ok, h_ok, sl, sv))
    return out

for with_z, label in ((False, "M1 naive only"), (True, "M2 naive + converted Z")):
    cnt = collections.Counter(); per_p = {}
    for p in P:
        fl = complete_flags(p, with_z)
        cnt["steps minute-complete"] += sum(f[0] for f in fl); cnt["hr minute-complete"] += sum(f[1] for f in fl)
        cnt["sleep present"] += sum(f[2] for f in fl); cnt["survey present"] += sum(f[3] for f in fl)
        cnt["steps+hr complete"] += sum(f[0] and f[1] for f in fl)
        full = sum(all(f) for f in fl); cnt["all four sources complete"] += full; per_p[p] = full
    print(f"[{label}] participant-days (of {len(P)*28}):", dict(cnt))
    print("   participants with all 28 days complete:", sum(1 for v in per_p.values() if v == 28), "| below 28:", {p: v for p, v in per_p.items() if v < 28})
partial = collections.Counter()
for kind in MIN:
    for p in P:
        for i in range(28):
            n = MIN[kind][p]["naive"][i]
            if 0 < n < 1440:
                partial[kind] += 1
print("participant-days with some but not all 1,440 minute slots (naive):", dict(partial))
print("participant-days with zero naive minute slots:", {k: sum(1 for p in P for i in range(28) if MIN[k][p]["naive"][i] == 0) for k in MIN})
reason_days = collections.Counter(); flagged_pd = set()
for kind in MIN:
    for p in P:
        for reason, idxs in MIN[kind][p]["flags"].items():
            reason_days[f"{kind}:{reason}"] += len(idxs)
            flagged_pd |= {(p, i) for i in idxs}
print("participant-days carrying quality flags, by reason:", dict(reason_days))
print("distinct participant-days with at least one quality flag:", len(flagged_pd))
clean = sum(1 for p in P for i, f in enumerate(complete_flags(p, True)) if all(f) and (p, i) not in flagged_pd)
print("participant-days complete (M2) AND free of the quality flags above:", clean, "of", len(P) * 28)

[M1 naive only] participant-days (of 1400): {'steps minute-complete': 1393, 'hr minute-complete': 1391, 'sleep present': 1400, 'survey present': 1400, 'steps+hr complete': 1391, 'all four sources complete': 1391}
   participants with all 28 days complete: 41 | below 28: {'P010': 27, 'P018': 27, 'P019': 27, 'P020': 27, 'P022': 27, 'P025': 27, 'P028': 27, 'P041': 27, 'P044': 27}
[M2 naive + converted Z] participant-days (of 1400): {'steps minute-complete': 1393, 'hr minute-complete': 1393, 'sleep present': 1400, 'survey present': 1400, 'steps+hr complete': 1393, 'all four sources complete': 1393}
   participants with all 28 days complete: 43 | below 28: {'P010': 27, 'P019': 27, 'P020': 27, 'P022': 27, 'P025': 27, 'P028': 27, 'P041': 27}
participant-days with some but not all 1,440 minute slots (naive): {'steps': 3, 'heart_rate': 3}
participant-days with zero naive minute slots: {'steps': 4, 'heart_rate': 6}
participant-days carrying quality flags, by reason: {'steps:_unit_field': 84, 'st

## 5. Design-option evidence: week boundary, cohort, freshness, skeleton source

In [6]:
print("study window weekday of first/last day:", DAYS[0].strftime("%A"), "/", DAYS[-1].strftime("%A"))
def blocks(key):
    c = collections.Counter(key(d) for d in DAYS)
    return list(c.values())
print("W1 study-relative 7-day blocks (days per week):", blocks(lambda d: (d - WIN_START).days // 7))
print("W2 ISO Monday-start weeks (days per week):", blocks(lambda d: d.isocalendar()[:2]))
print("W3 Sunday-start weeks (days per week):", blocks(lambda d: (d + timedelta(days=1)).isocalendar()[:2]))
cohort = collections.Counter(chrono.values())
print("chronotype counts:", dict(cohort))
full_days = {p: sum(all(f) for f in complete_flags(p, True)) for p in P}
for thr in (28, 26, 24, 21):
    kept = [p for p in P if full_days[p] >= thr]
    print(f"cohort filter: participants with >= {thr} complete days (M2): total={len(kept)} by chronotype={dict(collections.Counter(chrono[p] for p in kept))}")
sleep_days_by_c = collections.Counter(chrono[p] for p in P for _ in SLEEP[p])
print("sleep sessions by chronotype:", dict(sleep_days_by_c))
print("source candidates for a walking skeleton (rows | grain evidence):")
print("   participants.csv:", len(participants_rows), "| devices:", len(devices_rows), "| survey rows (enrolled):", sum(len(survey_days[p]) for p in P),
      "| sleep sessions:", sum(len(SLEEP[p]) for p in P), "| minute records per source on grid:", sum(m["slots"] for m in MIN["steps"].values()))
print("   survey is daily grain per participant (one row per participant-day):", all(len(survey_days[p]) == len(set(survey_days[p])) for p in survey_days) and len(survey_rows) == len({(r['participant_id'], r['date']) for r in survey_rows}))

study window weekday of first/last day: Thursday / Wednesday
W1 study-relative 7-day blocks (days per week): [7, 7, 7, 7]
W2 ISO Monday-start weeks (days per week): [4, 7, 7, 7, 3]
W3 Sunday-start weeks (days per week): [3, 7, 7, 7, 4]
chronotype counts: {'B': 22, 'A': 28}
cohort filter: participants with >= 28 complete days (M2): total=43 by chronotype={'B': 20, 'A': 23}
cohort filter: participants with >= 26 complete days (M2): total=50 by chronotype={'B': 22, 'A': 28}
cohort filter: participants with >= 24 complete days (M2): total=50 by chronotype={'B': 22, 'A': 28}
cohort filter: participants with >= 21 complete days (M2): total=50 by chronotype={'B': 22, 'A': 28}
sleep sessions by chronotype: {'B': 616, 'A': 784}
source candidates for a walking skeleton (rows | grain evidence):
   participants.csv: 50 | devices: 50 | survey rows (enrolled): 1400 | sleep sessions: 1400 | minute records per source on grid: 2009160
   survey is daily grain per participant (one row per participant-da

## Runtime

In [7]:
print(f"total notebook runtime: {time.perf_counter() - T0:.1f} s (harness limit 600 s)")

total notebook runtime: 41.5 s (harness limit 600 s)
